<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINT-Colab/blob/main/notebooks/04_embeddings_vinculos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 04 · Embeddings para resolução de entidades e comportamento coordenado

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINT-Colab)

Com o **multilingual-e5** transformamos bios e postagens em vetores. Isso permite achar contas que provavelmente são da mesma pessoa e textos quase idênticos espalhados por várias contas (sinal de comportamento inautêntico coordenado). Combinamos com similaridade de username, a mesma ideia do SÓSIA.

| | |
|---|---|
| **Hardware** | CPU (não precisa de GPU) |
| **Tempo estimado** | 3 minutos (CPU) |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINT-Colab.git /content/OSINT-Colab 2>/dev/null || git -C /content/OSINT-Colab pull -q
import sys
sys.path.insert(0, "/content/OSINT-Colab")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

In [ ]:
from sentence_transformers import SentenceTransformer
MODELO = "intfloat/multilingual-e5-base"
enc = SentenceTransformer(MODELO)

linhas = dados.carregar_jsonl("/content/OSINT-Colab/dados/demo/triagem_perfis.jsonl")
def campo(state, nome):
    for l in state.splitlines():
        if l.startswith(nome):
            return l.split(":", 1)[1].strip()
    return ""
perfis = [{"id": l["id"], "user": campo(l["state"], "Perfil analisado").lstrip("@"),
           "bio": campo(l["state"], "Bio"), "link": campo(l["state"], "Link na bio"),
           "humano": l["rotulo_humano"]["perfil"]} for l in linhas[:120]]
print(len(perfis), "perfis")

## 1. Vetores
O e5 pede o prefixo `query: ` (ou `passage: `). `normalize_embeddings=True` deixa o produto escalar igual ao cosseno.

In [ ]:
vet = enc.encode(["query: " + p["bio"] for p in perfis], normalize_embeddings=True)
sim = vet @ vet.T
print(sim.shape)

## 2. Pontuação de vínculo entre contas
`0.5 × bio + 0.3 × username + 0.2 × mesmo domínio`. Os pesos são um ponto de partida; ajuste com casos reais.

In [ ]:
from difflib import SequenceMatcher
from urllib.parse import urlparse

def dominio(url):
    return urlparse(url).netloc.lower()

def vinculo(i, j):
    a, b = perfis[i], perfis[j]
    s_user = SequenceMatcher(None, a["user"], b["user"]).ratio()
    s_dom = 1.0 if dominio(a["link"]) == dominio(b["link"]) else 0.0
    return 0.5 * float(sim[i, j]) + 0.3 * s_user + 0.2 * s_dom

pares = sorted(((vinculo(i, j), i, j) for i in range(len(perfis)) for j in range(i + 1, len(perfis))
                if perfis[i]["user"] != perfis[j]["user"]), reverse=True)
for s, i, j in pares[:10]:
    print(f"{s:.3f}  @{perfis[i]['user']:<28} @{perfis[j]['user']:<28} ({perfis[i]['humano']}/{perfis[j]['humano']})")

## 3. Agrupar contas (union-find, só stdlib)

In [ ]:
LIMIAR = 0.80
pai = list(range(len(perfis)))
def raiz(x):
    while pai[x] != x:
        pai[x] = pai[pai[x]]; x = pai[x]
    return x
for s, i, j in pares:
    if s < LIMIAR: break
    pai[raiz(i)] = raiz(j)

from collections import defaultdict
grupos = defaultdict(set)
for i, p in enumerate(perfis):
    grupos[raiz(i)].add(p["user"])
grupos = sorted((g for g in grupos.values() if len(g) > 1), key=len, reverse=True)
for g in grupos[:5]:
    print(len(g), sorted(g))

## 4. Distância do alvo oficial
Quanto mais parecido com `@bancoaurora` sem ser ele, mais suspeito.

In [ ]:
ALVO = "bancoaurora"
suspeitos = sorted({p["user"] for p in perfis if p["user"] != ALVO},
                   key=lambda u: SequenceMatcher(None, ALVO, u).ratio(), reverse=True)
for u in suspeitos[:10]:
    print(f"{SequenceMatcher(None, ALVO, u).ratio():.2f}  @{u}")

In [ ]:
import json, os
os.makedirs("/content/saidas", exist_ok=True)
with open("/content/saidas/grupos.json", "w", encoding="utf-8") as f:
    json.dump([sorted(g) for g in grupos], f, ensure_ascii=False, indent=2)
custodia.registrar("/content/saidas/manifesto.jsonl", "resolucao_entidades", "/content/saidas/grupos.json", {"modelo": MODELO, "limiar": LIMIAR})

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*